# Linguistic Features Training Notebook Ver2

🔹 Goal: Fine-tune a RoBERTa model on SemEval-2024 Task 8 Subtask A with linguistic features from Gemini 2.0 Flash.

🔹 Output: The linguistic model ver1 saved as "linguistic_model_ver2.pth".

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Setup & Install Dependencies

In [ ]:
%pip install transformers jsonlines tqdm

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from tqdm import tqdm
import jsonlines
import joblib
import numpy as np
import os
from transformers import RobertaTokenizer

## Step 2: Load Gemini-extracted CSV

In [ ]:
train_df = pd.read_csv('/content/drive/MyDrive/SemEval-2024 Task 8/Gemini Features/gemini_linguistic_extraction_train.csv')
dev_df = pd.read_csv('/content/drive/MyDrive/SemEval-2024 Task 8/Gemini Features/gemini_linguistic_extraction_dev.csv')

## Step 3: Preprocess Features

### 🔹 3.1 Prepare and Load the Scaler

In [ ]:
# Load scaler
scaler = joblib.load("/content/drive/MyDrive/SemEval-2024 Task 8/models/scaler.pkl")

### 🔹 3.2 Preprocess Features

In [ ]:
# Binary columns: convert and fill missing
binary_cols = ['HasParticipialPhrase', 'HasHangingParticiple', 'UsesPassiveVoice', 'SentenceStartRepetition']
train_df[binary_cols] = train_df[binary_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
dev_df[binary_cols] = dev_df[binary_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# Numeric columns (only actual numeric features)
num_cols = [
    'NumIndependentClauses', 'NumDependentClauses', 'AvgSentenceLength',
    'SentenceLengthVariance', 'ClauseEmbeddingDepth',
    'TypeTokenRatio', 'NumIdioms'
]

# Normalize
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
dev_df[num_cols] = scaler.transform(dev_df[num_cols])

# Drop the 'SentenceStructure' column
train_df = train_df.drop(columns=['Index', 'SentenceStructure'])
dev_df = dev_df.drop(columns=['Index', 'SentenceStructure'])

## Step 4: Create Dataset Class

In [ ]:
class GeminiTextDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, gemini_df, tokenizer, max_length=512):
        self.data = []
        self.max_length = max_length
        self.df = pd.read_json(jsonl_file, lines=True)
        self.features = gemini_df
        self.tokenizer = tokenizer

        # Load texts and labels
        with jsonlines.open(jsonl_file) as reader:
            entries = list(reader)

        assert len(entries) == len(gemini_df), "Mismatch between JSONL and CSV rows"

        for i in tqdm(range(len(entries)), desc=f"Loading {jsonl_file} with Gemini features"):
            obj = entries[i]
            text = obj["text"]
            label = obj["label"]

            # Tokenize text
            encoding = tokenizer(
                text,
                truncation=True,
                padding='max_length',
                max_length=max_length,
                return_tensors='pt'
            )

            # Get the corresponding Gemini features (as numpy array)
            feats = gemini_df.iloc[i].drop("Index", errors="ignore").values.astype(float)

            self.data.append((encoding, feats, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoding, ling_feats, label = self.data[idx]
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'ling_feats': torch.tensor(ling_feats, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

## Step 5: Load Tokenizer, Data, and Dataloaders

### 🔹 5.1 Create Train & Dev Datasets

In [ ]:
train_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_train_monolingual.jsonl"
dev_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_dev_monolingual.jsonl"

In [ ]:
# # Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Define paths
train_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_train_monolingual.jsonl"
dev_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_dev_monolingual.jsonl"

# Create datasets
# train_dataset = GeminiTextDataset(train_file, train_df, tokenizer)
# dev_dataset = GeminiTextDataset(dev_file, dev_df, tokenizer)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Loading /content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_train_monolingual.jsonl with Gemini features: 100%|██████████| 119757/119757 [13:32<00:00, 147.34it/s]
Loading /content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_dev_monolingual.jsonl with Gemini features: 100%|██████████| 5000/5000 [00:25<00:00, 196.12it/s]


### 🔹 5.2 Save the Train and Dev Datasets

In [ ]:
# Save the datasets for the first time run
# torch.save(train_dataset, "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/train_dataset_ver2.pt")
# torch.save(dev_dataset, "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/dev_dataset_ver2.pt")

### 🔹 5.3 Load Train & Dev Datasets

In [ ]:
train_dataset = torch.load(
    "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/train_dataset_ver2.pt",
    weights_only=False
)
dev_dataset = torch.load(
    "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/dev_dataset_ver2.pt",
    weights_only=False
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)

## Step 6: Define and Initialize RobertaWithLingFeatures Model

### 🔹 6.1 Define Model

In [ ]:
import torch.nn as nn
class RobertaWithGeminiFeatures(nn.Module):
    def __init__(self, ling_feat_dim, num_labels=2):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size + ling_feat_dim, num_labels)

    def forward(self, input_ids, attention_mask, ling_feats):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]  # [CLS] token
        combined = torch.cat((cls_embedding, ling_feats), dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        return logits

### 🔹 6.2 Initialize Model

In [ ]:
ling_feat_dim = train_df.shape[1] - 2  # Exclude the 'Index' and 'SentenceStructure' column
model = RobertaWithGeminiFeatures(ling_feat_dim)

In [ ]:
train_df.shape[1]

13

In [ ]:
train_df

,Index,SentenceStructure,NumIndependentClauses,NumDependentClauses,HasParticipialPhrase,HasHangingParticiple,AvgSentenceLength,SentenceLengthVariance,UsesPassiveVoice,ClauseEmbeddingDepth,TypeTokenRatio,NumIdioms,SentenceStartRepetition
0,0,Compound-Complex,19,13,1,0.0,16.4600,164.210,1,1.0,0.530,1,1
1,1,Compound-Complex,31,18,1,0.0,16.8200,204.480,1,1.0,0.490,0,1
2,2,Compound-Complex,28,20,1,0.0,17.1000,119.170,1,1.0,0.500,0,1
3,3,Compound-Complex,26,16,1,0.0,15.4000,128.260,0,1.0,0.560,0,1
4,4,Compound-Complex,20,16,1,0.0,16.9400,176.070,1,1.0,0.550,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
119752,119752,Compound,5,2,1,0.0,24.2000,38.700,0,1.0,0.736,0,0
119753,119753,Compound-Complex,30,26,1,0.0,25.3929,212.647,1,3.0,0.621,1,0
119754,119754,Compound-Complex,7,10,1,0.0,27.5294,136.797,1,2.0,0.547,0,0
119755,119755,Compound-Complex,9,8,1,0.0,28.8333,125.972,1,2.0,0.560,0,0


## Step 7: Training Loop with Progress Bar

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Training]")

    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ling_feats = batch["ling_feats"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, ling_feats)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())

        progress.set_postfix(loss=loss.item())

    # Compute metrics
    avg_loss = total_loss / len(train_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')

    print(f"\nEpoch {epoch+1} -- Avg Dev Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f} | Macro F1: {f1_macro:.4f}")

Epoch 1/3 [Training]: 100%|██████████| 7485/7485 [38:02<00:00,  3.28it/s, loss=3.27e-5]



Epoch 1 -- Avg Dev Loss: 0.0140 | Accuracy: 0.9956 | Macro F1: 0.9956


Epoch 2/3 [Training]: 100%|██████████| 7485/7485 [38:01<00:00,  3.28it/s, loss=4.85e-5]



Epoch 2 -- Avg Dev Loss: 0.0084 | Accuracy: 0.9972 | Macro F1: 0.9972


Epoch 3/3 [Training]: 100%|██████████| 7485/7485 [38:01<00:00,  3.28it/s, loss=0.000149]


Epoch 3 -- Avg Dev Loss: 0.0059 | Accuracy: 0.9983 | Macro F1: 0.9983


## Step 8: Save Model and Tokenizer

In [ ]:
# Set directory for saving linguistic model
linguistic_model_path = "/content/drive/MyDrive/SemEval-2024 Task 8/models/linguistic models"
os.makedirs(linguistic_model_path, exist_ok=True)

# Save model checkpoint
model_save_path = os.path.join(linguistic_model_path, "linguistic_model_ver2.pth")
torch.save(model.state_dict(), model_save_path)

print(f"Linguistic model saved at: {model_save_path}")

Linguistic model saved at: /content/drive/MyDrive/SemEval-2024 Task 8/models/linguistic models/linguistic_model_ver2.pth
